In [2]:
import qiskit
import qiskit_aer

print("Qiskit version:", qiskit.__version__)
print("Qiskit Aer version:", qiskit_aer.__version__)

print("4-bit Adder-Subtractor Project Started!")

Qiskit version: 2.5.2
Qiskit Aer version: 0.17.2
4-bit Adder-Subtractor Project Started!


In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

simulator = AerSimulator()

print("Libraries imported successfully!")

Libraries imported successfully!


In [8]:
def create_4bit_baseline_adder_subtractor(a, b, mode):

    # 4-bit input registers
    A = QuantumRegister(4, "A")
    B = QuantumRegister(4, "B")

    # Operation mode: 0 = Addition, 1 = Subtraction
    M = QuantumRegister(1, "M")

    # 4-bit result register
    R = QuantumRegister(4, "R")

    # Carry registers: initial carry + 4 carry outputs
    C = QuantumRegister(5, "C")

    # Classical result registers
    result_bits = ClassicalRegister(4, "result")
    final_carry = ClassicalRegister(1, "carry")

    qc = QuantumCircuit(
        A, B, M, R, C,
        result_bits, final_carry
    )

    # Load input A
    for i in range(4):
        if (a >> i) & 1:
            qc.x(A[i])

    # Load input B
    for i in range(4):
        if (b >> i) & 1:
            qc.x(B[i])

    # Set operation mode
    if mode == 1:
        qc.x(M[0])

    # Initial carry = mode
    qc.cx(M[0], C[0])

    # Ripple-carry full adder stages
    for i in range(4):

        # B' = B XOR mode
        qc.cx(M[0], B[i])

        # Sum = A XOR B' XOR carry-in
        qc.cx(A[i], R[i])
        qc.cx(B[i], R[i])
        qc.cx(C[i], R[i])

        # Carry calculation using majority logic
        # Carry calculation using majority logic
        qc.ccx(A[i], B[i], C[i + 1])
        qc.ccx(A[i], C[i], C[i + 1])
        qc.ccx(B[i], C[i], C[i + 1])

        # Restore original B
        qc.cx(M[0], B[i])

    # Measure result
    for i in range(4):
        qc.measure(R[i], result_bits[i])

    # Measure final carry
    qc.measure(C[4], final_carry[0])

    return qc


print("4-bit baseline circuit function created!")

4-bit baseline circuit function created!


In [9]:
def test_4bit_circuit(a, b, mode):

    qc = create_4bit_baseline_adder_subtractor(a, b, mode)

    result = simulator.run(
        qc,
        shots=1
    ).result()

    counts = result.get_counts()

    measured = next(iter(counts))

    # Remove spaces between classical registers
    parts = measured.split()

    # Classical registers are displayed in reverse register order
    carry_value = int(parts[0], 2)
    result_value = int(parts[1], 2)

    if mode == 0:
        expected = (a + b) % 16
        expected_carry = (a + b) // 16
        operation = "Addition"
    else:
        expected = (a - b) % 16
        expected_carry = int(a >= b)
        operation = "Subtraction"

    print(f"\nOperation: {operation}")
    print(f"A = {a}, B = {b}")
    print(f"Expected result: {expected}")
    print(f"Actual result:   {result_value}")
    print(f"Expected carry:  {expected_carry}")
    print(f"Actual carry:    {carry_value}")

    if result_value == expected and carry_value == expected_carry:
        print("PASS")
    else:
        print("FAIL")

In [10]:
test_4bit_circuit(5, 3, 0)
test_4bit_circuit(10, 6, 0)
test_4bit_circuit(10, 5, 1)
test_4bit_circuit(5, 10, 1)


Operation: Addition
A = 5, B = 3
Expected result: 8
Actual result:   8
Expected carry:  0
Actual carry:    0
PASS

Operation: Addition
A = 10, B = 6
Expected result: 0
Actual result:   0
Expected carry:  1
Actual carry:    1
PASS

Operation: Subtraction
A = 10, B = 5
Expected result: 5
Actual result:   5
Expected carry:  1
Actual carry:    1
PASS

Operation: Subtraction
A = 5, B = 10
Expected result: 11
Actual result:   11
Expected carry:  0
Actual carry:    0
PASS


In [11]:
total_tests = 0
passed_tests = 0
failed_tests = 0

for mode in [0, 1]:
    for a in range(16):
        for b in range(16):

            qc = create_4bit_baseline_adder_subtractor(a, b, mode)

            result = simulator.run(qc, shots=1).result()
            counts = result.get_counts()
            measured = next(iter(counts)).split()

            carry_value = int(measured[0], 2)
            result_value = int(measured[1], 2)

            if mode == 0:
                expected_result = (a + b) % 16
                expected_carry = (a + b) // 16
            else:
                expected_result = (a - b) % 16
                expected_carry = int(a >= b)

            total_tests += 1

            if (result_value == expected_result and
                    carry_value == expected_carry):
                passed_tests += 1
            else:
                failed_tests += 1
                print("FAILED:", a, b, mode)

print("\nBASELINE VERIFICATION")
print("Total tests:", total_tests)
print("Passed:", passed_tests)
print("Failed:", failed_tests)


BASELINE VERIFICATION
Total tests: 512
Passed: 512
Failed: 0


In [12]:
def calculate_4bit_metrics(qc):

    # Remove final measurements for gate analysis
    circuit = qc.remove_final_measurements(inplace=False)

    counts = circuit.count_ops()

    cnot_count = counts.get("cx", 0)
    toffoli_count = counts.get("ccx", 0)
    x_count = counts.get("x", 0)

    total_gates = sum(counts.values())

    # Our chosen logical quantum cost model
    quantum_cost = cnot_count + (5 * toffoli_count)

    metrics = {
        "Qubits": circuit.num_qubits,
        "CNOT": cnot_count,
        "Toffoli": toffoli_count,
        "X": x_count,
        "Total Gates": total_gates,
        "Depth": circuit.depth(),
        "Quantum Cost": quantum_cost
    }

    return metrics

In [13]:
baseline_circuit = create_4bit_baseline_adder_subtractor(
    5, 3, 0
)

baseline_metrics = calculate_4bit_metrics(baseline_circuit)

print("4-BIT BASELINE PERFORMANCE")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value}")

4-BIT BASELINE PERFORMANCE
Qubits: 18
CNOT: 21
Toffoli: 12
X: 4
Total Gates: 37
Depth: 25
Quantum Cost: 81


## 4-Bit Baseline Adder-Subtractor: Performance Results

- Qubits: 18
- CNOT gates: 21
- Toffoli gates: 12
- X gates: 4
- Total gates: 37
- Circuit depth: 25
- Quantum cost: 81

The baseline circuit successfully passed all 512 combinations
of 4-bit addition and subtraction.